# B2 — One-Hot by Hand

---

## 🔍 Problem-এ কী চাওয়া হয়েছে?

আমাদের কাছে একটি `City` column আছে:

$$\text{Cities} = [\text{Dhaka, Chattogram, Dhaka, Rajshahi, Rajshahi}]$$

এই একটি column থেকে **তিনটি binary column** তৈরি করতে হবে:

| তৈরি করতে হবে | নিয়ম |
|---|---|
| `City_Dhaka` | যে row-এ Dhaka আছে → **1**, বাকি সব → **0** |
| `City_Chattogram` | যে row-এ Chattogram আছে → **1**, বাকি সব → **0** |
| `City_Rajshahi` | যে row-এ Rajshahi আছে → **1**, বাকি সব → **0** |

শুধু 0 ও 1 ব্যবহার করে হাতে হিসাব করতে হবে।


---

## 🎯 এই কাজ থেকে আমরা কী অর্জন করতে পারব?

- **One-Hot Encoding**-এর ভেতরের mechanics হাতে করে বুঝব।
- প্রতিটি row-এ **ঠিক একটিমাত্র column-এ 1** থাকে কেন — সেটার intuition পাকা হবে।
- কেন Nominal data-তে One-Hot Encoding দরকার — ML model-এর দৃষ্টিকোণ থেকে বুঝব।
- `pd.get_dummies()` আসলে হাতের হিসাবের কোন কাজটি automatically করে সেটা দেখব।


---

## 🧠 আমরা যা শিখেছি, সেই আলোকে কীভাবে চিন্তা করতে হবে?

### One-Hot Encoding কেন দরকার?

`City` column-এ text আছে — ML model text বোঝে না, শুধু সংখ্যা বোঝে।
সরাসরি Dhaka=1, Chattogram=2, Rajshahi=3 দিলে model ভাববে Rajshahi > Chattogram > Dhaka — যা সম্পূর্ণ ভুল।

One-Hot Encoding-এ প্রতিটি city আলাদা binary column পায় — কোনো ভুল ক্রম তৈরি হয় না।

### হাতের নিয়ম:

প্রতিটি row-এর জন্য:
> **"এই row-এর City কি X?"** → হ্যাঁ = **1**, না = **0**

### হাতে তৈরি table:

| Row | City | City_Dhaka | City_Chattogram | City_Rajshahi |
|---|---|---|---|---|
| 1 | Dhaka | **1** | 0 | 0 |
| 2 | Chattogram | 0 | **1** | 0 |
| 3 | Dhaka | **1** | 0 | 0 |
| 4 | Rajshahi | 0 | 0 | **1** |
| 5 | Rajshahi | 0 | 0 | **1** |

**গুরুত্বপূর্ণ pattern:** প্রতিটি row-এ সবসময় ঠিক **একটিমাত্র column-এ 1** এবং বাকি সব **0**।


---

## 🛠️ Problem Solve করার Approach

**Step 1:** City data তৈরি করা।

**Step 2:** হাতে হিসাব — manually তিনটি binary column তৈরি করা।

**Step 3:** `pd.get_dummies()` দিয়ে verify করা।

**Step 4:** দুটো result মিলিয়ে দেখা।


## Step 1: Data তৈরি করা

In [2]:
import pandas as pd
import numpy as np

cities = ['Dhaka', 'Chattogram', 'Dhaka', 'Rajshahi', 'Rajshahi']
df = pd.DataFrame({'City': cities})

print("Original data:")
print(df.to_string())
print()
print(f"Unique cities: {df['City'].unique()}")
print(f"Total rows   : {len(df)}")


Original data:
         City
0       Dhaka
1  Chattogram
2       Dhaka
3    Rajshahi
4    Rajshahi

Unique cities: ['Dhaka' 'Chattogram' 'Rajshahi']
Total rows   : 5


`pd.DataFrame({'City': cities})` → list থেকে single-column DataFrame তৈরি।
`df['City'].unique()` → column-এর সব unique value দেখায়।


---

## Step 2: হাতে One-Hot Encoding তৈরি করা

প্রতিটি column-এর জন্য: **সেই city হলে 1, না হলে 0**।


In [3]:
df['City_Dhaka']      = (df['City'] == 'Dhaka').astype(int)
df['City_Chattogram'] = (df['City'] == 'Chattogram').astype(int)
df['City_Rajshahi']   = (df['City'] == 'Rajshahi').astype(int)

print("── Manual One-Hot Encoding ──")
print(df.to_string(index=False))


── Manual One-Hot Encoding ──
      City  City_Dhaka  City_Chattogram  City_Rajshahi
     Dhaka           1                0              0
Chattogram           0                1              0
     Dhaka           1                0              0
  Rajshahi           0                0              1
  Rajshahi           0                0              1


`df['City'] == 'Dhaka'` → প্রতিটি row-এ True/False দেয়।
`.astype(int)` → True → **1**, False → **0**-এ convert করে।

এই তিন লাইন কোডই হাতের হিসাবের তিনটি column তৈরির কাজটি করছে।


In [4]:
ohe_cols = ['City_Dhaka', 'City_Chattogram', 'City_Rajshahi']
row_sums = df[ohe_cols].sum(axis=1)

print("Row-wise sum of OHE columns (should always be 1):")
print(row_sums.to_string())
print()
print(f"All rows sum to 1: {(row_sums == 1).all()}")


Row-wise sum of OHE columns (should always be 1):
0    1
1    1
2    1
3    1
4    1

All rows sum to 1: True


`df[ohe_cols].sum(axis=1)` → প্রতিটি row-এর তিনটি column-এর যোগফল।
**সবসময় 1 হবে** — কারণ প্রতিটি row শুধু একটি city-র জন্য 1, বাকি দুটো 0।
এটি One-Hot Encoding-এর সবচেয়ে গুরুত্বপূর্ণ property।


---

## Step 3: `pd.get_dummies()` দিয়ে Verify করা


In [5]:
original_df = pd.DataFrame({'City': cities})
auto_encoded = pd.get_dummies(original_df['City'], prefix='City').astype(int)

print("── Auto One-Hot via pd.get_dummies() ──")
print(auto_encoded.to_string(index=False))


── Auto One-Hot via pd.get_dummies() ──
 City_Chattogram  City_Dhaka  City_Rajshahi
               0           1              0
               1           0              0
               0           1              0
               0           0              1
               0           0              1


`pd.get_dummies()` internally exactly same কাজ করে যা আমরা হাতে করেছি।
`prefix='City'` → column-এর নাম `City_Dhaka` এর মতো হয়।
`.astype(int)` → pandas কখনো bool দেয়, তাই int-এ convert।


## Step 4: হাতের হিসাব vs `get_dummies()` — মিলিয়ে দেখা

In [6]:
manual = df[ohe_cols].reset_index(drop=True)
auto   = auto_encoded[ohe_cols].reset_index(drop=True)

match = manual.equals(auto)
print(f"Manual encoding matches get_dummies(): {match}")
print()
print("Final encoded dataset:")
final = pd.concat([pd.DataFrame({'City': cities}), manual], axis=1)
print(final.to_string(index=False))


Manual encoding matches get_dummies(): True

Final encoded dataset:
      City  City_Dhaka  City_Chattogram  City_Rajshahi
     Dhaka           1                0              0
Chattogram           0                1              0
     Dhaka           1                0              0
  Rajshahi           0                0              1
  Rajshahi           0                0              1


`manual.equals(auto)` → দুটো DataFrame হুবহু একই কিনা check করে।
`True` দেখালে confirm — হাতের হিসাব ও `pd.get_dummies()` একই কাজ করে। ✅

**Final dataset-এ:**
- Original `City` text column সরিয়ে দিলে তিনটি numeric binary column থাকবে।
- ML model এই তিনটি column সরাসরি ব্যবহার করতে পারবে।


---